In [1]:
# Must be the FIRST cell, before anything else loads
options(timeout = 600000000)
install.packages("rlang", repos = "https://cran.r-project.org")
install.packages("vctrs", repos = "https://cran.r-project.org")
install.packages("lifecycle", repos = "https://cran.r-project.org")
install.packages("dplyr", repos = "https://cran.r-project.org")
devtools::install_github("dmcable/spacexr", build_vignettes = FALSE, upgrade = "never")
library(spacexr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages("rlang", repos = "https://cran.r-project.org"):
“installation of package ‘rlang’ had non-zero exit status”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rlang’


Warning message in install.packages("vctrs", repos = "https://cran.r-project.org"):
“installation of package ‘rlang’ had non-zero exit status”
Warning message in install.packages("vctrs", repos = "https://cran.r-project.org"):
“installation of package ‘vctrs’ had non-zero exit status”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Warning message in install.packages("lifecycle", repos = "https://cran.r-project.org"):
“installation of package ‘lifecycle’ had non-zero exit status”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘lifecycle’, ‘

── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/RtmpRCA705/remotes2513cd29d/dmcable-spacexr-9f5dc33/DESCRIPTION’ ... OK
* preparing ‘spacexr’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
Omitted ‘LazyData’ from DESCRIPTION
* building ‘spacexr_2.2.1.tar.gz’



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [1]:
library(spacexr)
library(Seurat)
library(Matrix)

# ── CONFIG ───────────────────────────────────────────────────────
csv_root   <- "/kaggle/input/notebooks/wanianaeem/scvi-latent-embeddings/scVI_counts"
sc_rds     <- "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/scRNA-seq_Data_post_qc.rds"
output_dir <- "/kaggle/working/rctd_weights"
dir.create(output_dir, showWarnings = FALSE)

sample_ids <- c(
  "IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
  "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4"
)

# ── STEP 1: Load scRNA reference ─────────────────────────────────
cat("Loading scRNA reference...\n")
sc <- readRDS(sc_rds)
cell_type_col <- "celltype_new1"

# ── STEP 2: Build Reference ONCE ────────────────────────────────
counts_ref <- GetAssayData(sc, assay = "RNA", layer = "counts")
cell_types <- as.factor(sc@meta.data[[cell_type_col]])
names(cell_types) <- colnames(counts_ref)
nUMI_ref <- colSums(counts_ref)

reference <- Reference(counts_ref, cell_types, nUMI_ref)
cat("Reference built:", nlevels(cell_types), "cell types\n")
print(levels(cell_types))
rm(sc); gc()

# ── STEP 3: Load coords ONCE ─────────────────────────────────────
coords_all <- read.csv(
  "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/spot_spatial_coordinates.csv",
  row.names = 1
)

# ── STEP 4: Per-sample loop ──────────────────────────────────────
for (sid in sample_ids) {
  out_path <- file.path(output_dir, paste0(sid, "_rctd_weights.csv"))
  if (file.exists(out_path)) { cat("Skipping", sid, "\n"); next }

  cat("\n── Processing:", sid, "──\n")

  # Load count matrix (genes × barcodes)
  csv_path   <- file.path(csv_root, paste0(sid, ".csv"))
  df         <- read.csv(csv_path, row.names = 1, check.names = FALSE)
  counts_mat <- Matrix(as.matrix(df), sparse = TRUE)
  cat("  Counts:", nrow(counts_mat), "genes ×", ncol(counts_mat), "barcodes\n")

  # Filter coords to this sample
  sample_mask <- coords_all$image == sid
  coords      <- coords_all[sample_mask, c("imagecol", "imagerow")]
  colnames(coords) <- c("x", "y")
  rownames(coords) <- rownames(coords_all)[sample_mask]
  cat("  Coord spots:", nrow(coords), "\n")

  # Align barcodes
  common <- intersect(colnames(counts_mat), rownames(coords))
  cat("  Common barcodes:", length(common), "\n")
  counts_mat <- counts_mat[, common]
  coords     <- coords[common, ]
  nUMI       <- colSums(counts_mat)

  # Run RCTD
  spatial_obj <- SpatialRNA(coords, counts_mat, nUMI)
  myRCTD      <- create.RCTD(spatial_obj, reference, max_cores = 2)
  myRCTD      <- run.RCTD(myRCTD, doublet_mode = "full")

  # Normalize weights → proportions
  weights_df <- as.data.frame(as.matrix(normalize_weights(myRCTD@results$weights)))
  cat("  Output:", nrow(weights_df), "spots ×", ncol(weights_df), "cell types\n")
  write.csv(weights_df, out_path, row.names = TRUE)

  rm(myRCTD, spatial_obj, counts_mat, df, weights_df, coords); gc()
}

cat("\n✓ Done.\n")

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




Loading scRNA reference...


Warning message in Reference(counts_ref, cell_types, nUMI_ref):
“Reference: number of cells per cell type is 33686, larger than maximum allowable of 10000. Downsampling number of cells to: 10000”


Reference built: 15 cell types
 [1] "B cells"                 "C1Q-TAM"                
 [3] "CD4+ cells"              "CD8-NK cells"           
 [5] "DCs"                     "Endothelial cells"      
 [7] "FCN1-TAM"                "Hepatocytes"            
 [9] "iCAF"                    "myCAF"                  
[11] "Normal Epithelial cells" "Proliferative T cells"  
[13] "PVL"                     "SPP1-TAM"               
[15] "Tumor Epithelial cells" 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,3644769,194.7,6772571,361.7,4599124,245.7
Vcells,493168368,3762.6,1711985839,13061.5,1875108294,14306.0


...

  Output: 4354 spots × 15 cell types

── Processing: IU_PDA_T4 ──
  Counts: 17893 genes × 3621 barcodes
  Coord spots: 3621 
  Common barcodes: 3621 


Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 69710

process_cell_type_info: number of genes in reference: 30782




                B cells                 C1Q-TAM              CD4+ cells 
                   1795                    3207                    3256 
           CD8-NK cells                     DCs       Endothelial cells 
                   6474                    2046                   10000 
               FCN1-TAM             Hepatocytes                    iCAF 
                   3768                    1953                    2131 
                  myCAF Normal Epithelial cells   Proliferative T cells 
                   2357                   10000                    1051 
                    PVL                SPP1-TAM  Tumor Epithelial cells 
                   8831                    2841                   10000 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.5 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.3 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.0 GiB”
Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 2.3 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B cells found DE genes: 74

get_de_genes: C1Q-TAM found DE genes: 227

get_de_genes: CD4+ cells found DE genes: 300

get_de_genes: CD8-NK cells found DE genes: 255

get_de_genes: DCs found DE genes: 150

get_de_genes: Endothelial cells found DE genes: 248

get_de_genes: FCN1-TAM found DE genes: 229

get_de_genes: Hepatocytes found DE genes: 186

get_de_genes: iCAF found DE genes: 229

get_de_genes: myCA

  Output: 3621 spots × 15 cell types

✓ Done.


In [3]:
import os, glob
import pandas as pd
import torch
from tqdm.auto import tqdm

CONFIG = dict(
    rctd_root   = "/kaggle/input/datasets/wanianaeem/rctd-weights",
    png_root    = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches",
    coords_path = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/spot_spatial_coordinates.csv",
    output_root = "/kaggle/working/Features/RCTD",
    samples = [
        "IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
        "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4",
    ],
)

os.makedirs(CONFIG["output_root"], exist_ok=True)

# Load coords once — has both barcode (index) and row/col
coords_all = pd.read_csv(CONFIG["coords_path"], index_col=0)
print("Coords columns:", coords_all.columns.tolist())
print("Coords index (first 3):", coords_all.index[:3].tolist())

# Load all RCTD CSVs
all_dfs = {}
for sid in CONFIG["samples"]:
    df = pd.read_csv(os.path.join(CONFIG["rctd_root"], f"{sid}_rctd_weights.csv"), index_col=0)
    all_dfs[sid] = df

all_cell_types = sorted(set(ct for df in all_dfs.values() for ct in df.columns))
n_cell_types   = len(all_cell_types)
print(f"\nCell types ({n_cell_types}): {all_cell_types}")

total_saved, total_missing, total_failed = 0, 0, 0

for sid in CONFIG["samples"]:
    print("=" * 70)
    print(f"Processing: {sid}")
    print("=" * 70)

    out_dir = os.path.join(CONFIG["output_root"], sid)
    os.makedirs(out_dir, exist_ok=True)

    df = all_dfs[sid].reindex(columns=all_cell_types, fill_value=0.0).astype("float32")

    # Build (row, col) → barcode lookup from coords for this sample
    sample_coords = coords_all[coords_all["image"] == sid]
    # key: (row, col) → RCTD barcode (PDACH_XX_... format)
    rowcol_to_barcode = {
        (int(r["row"]), int(r["col"])): idx
        for idx, r in sample_coords.iterrows()
    }
    print(f"  Coord spots: {len(rowcol_to_barcode)}")

    patch_files = sorted(glob.glob(os.path.join(CONFIG["png_root"], sid, "*.png")))
    print(f"  Found {len(patch_files)} patches")

    existing = set(os.path.splitext(f)[0] for f in os.listdir(out_dir) if f.endswith(".pt"))
    to_process = [p for p in patch_files
                  if os.path.splitext(os.path.basename(p))[0] not in existing]
    print(f"  Processing {len(to_process)} new patches (existing: {len(existing)})")

    saved = missing = failed = 0

    for png_path in tqdm(to_process, desc=sid):
        patch_name = os.path.splitext(os.path.basename(png_path))[0]
        # patch_name: IU_PDA_HM11_patch-000001_50_102 → row=50, col=102
        parts = patch_name.split("_")
        try:
            row, col = int(parts[-2]), int(parts[-1])
        except (ValueError, IndexError):
            missing += 1
            continue

        barcode = rowcol_to_barcode.get((row, col))
        if barcode is None or barcode not in df.index:
            missing += 1
            continue

        try:
            emb = torch.tensor(df.loc[barcode].values, dtype=torch.float32)
            torch.save(emb, os.path.join(out_dir, f"{patch_name}.pt"))
            saved += 1
        except Exception as e:
            failed += 1

    print(f"  Saved   : {saved} / {len(to_process)}")
    if missing: print(f"  Missing : {missing}")
    if failed:  print(f"  Failed  : {failed}")
    total_saved += saved; total_missing += missing; total_failed += failed

print("\n" + "=" * 70)
print(f"TOTAL SAVED   : {total_saved}")
print(f"TOTAL MISSING : {total_missing}")
print(f"TOTAL FAILED  : {total_failed}")

print("\nPer-sample .pt counts:")
for sid in CONFIG["samples"]:
    count = len(glob.glob(os.path.join(CONFIG["output_root"], sid, "*.pt")))
    print(f"  • {sid:<20} {count} embeddings")

sample_check = CONFIG["samples"][0]
pt_files = glob.glob(os.path.join(CONFIG["output_root"], sample_check, "*.pt"))
if pt_files:
    emb = torch.load(pt_files[0], map_location="cpu")
    print(f"\nSpot-check — {os.path.basename(pt_files[0])}")
    print(f"  Shape  : {tuple(emb.shape)}")
    print(f"  Dtype  : {emb.dtype}")
    print(f"  First 5: {emb[:5].tolist()}")

Coords columns: ['tissue', 'row', 'col', 'imagerow', 'imagecol', 'image', 'patient']
Coords index (first 3): ['PDACH_12_AAACAAGTATCTCCCA-1', 'PDACH_12_AAACACCAATAACTGC-1', 'PDACH_12_AAACCGGGTAGGTACC-1']

Cell types (15): ['B cells', 'C1Q-TAM', 'CD4+ cells', 'CD8-NK cells', 'DCs', 'Endothelial cells', 'FCN1-TAM', 'Hepatocytes', 'Normal Epithelial cells', 'PVL', 'Proliferative T cells', 'SPP1-TAM', 'Tumor Epithelial cells', 'iCAF', 'myCAF']
Processing: IU_PDA_HM11
  Coord spots: 3931
  Found 3931 patches
  Processing 3931 new patches (existing: 0)


IU_PDA_HM11:   0%|          | 0/3931 [00:00<?, ?it/s]

  Saved   : 3931 / 3931
Processing: IU_PDA_HM13
  Coord spots: 2182
  Found 2182 patches
  Processing 2182 new patches (existing: 0)


IU_PDA_HM13:   0%|          | 0/2182 [00:00<?, ?it/s]

  Saved   : 2182 / 2182
Processing: IU_PDA_T1
  Coord spots: 3530
  Found 3530 patches
  Processing 3530 new patches (existing: 0)


...

  Saved   : 2777 / 2777
Processing: IU_PDA_T3
  Coord spots: 4354
  Found 4354 patches
  Processing 4354 new patches (existing: 0)


IU_PDA_T3:   0%|          | 0/4354 [00:00<?, ?it/s]

  Saved   : 4354 / 4354
Processing: IU_PDA_T4
  Coord spots: 3621
  Found 3621 patches
  Processing 3621 new patches (existing: 0)


IU_PDA_T4:   0%|          | 0/3621 [00:00<?, ?it/s]

  Saved   : 3621 / 3621

TOTAL SAVED   : 20395
TOTAL MISSING : 0
TOTAL FAILED  : 0

Per-sample .pt counts:
  • IU_PDA_HM11          3931 embeddings
  • IU_PDA_HM13          2182 embeddings
  • IU_PDA_T1            3530 embeddings
  • IU_PDA_T11           2777 embeddings
  • IU_PDA_T3            4354 embeddings
  • IU_PDA_T4            3621 embeddings

Spot-check — IU_PDA_HM11_patch-003135_64_46.pt
  Shape  : (15,)
  Dtype  : torch.float32
  First 5: [0.02696351148188114, 0.12956523895263672, 0.0714607983827591, 5.540417623706162e-05, 5.540417623706162e-05]


In [2]:
import pandas as pd
import glob
import os

sid = "IU_PDA_HM11"
rctd_dir = "/kaggle/input/datasets/wanianaeem/rctd-weights"
png_root = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches"

# RCTD CSV index
df = pd.read_csv(f"{rctd_dir}/{sid}_rctd_weights.csv", index_col=0)
print("RCTD barcodes (first 3):")
print(df.index[:3].tolist())

# PNG patch stems
patches = sorted(glob.glob(os.path.join(png_root, sid, "*.png")))
print("\nPNG patch stems (first 3):")
print([os.path.splitext(os.path.basename(p))[0] for p in patches[:3]])

RCTD barcodes (first 3):
['PDACH_10_AAACAAGTATCTCCCA-1', 'PDACH_10_AAACACCAATAACTGC-1', 'PDACH_10_AAACAGAGCGACTCCT-1']

PNG patch stems (first 3):
['IU_PDA_HM11_patch-000001_50_102', 'IU_PDA_HM11_patch-000002_59_19', 'IU_PDA_HM11_patch-000003_14_94']


In [4]:
import zipfile
import glob
import os

output_root = "/kaggle/working/Features/RCTD"
zip_path    = "/kaggle/working/RCTD_embeddings.zip"

print("Zipping...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pt_file in glob.glob(os.path.join(output_root, "**", "*.pt"), recursive=True):
        arcname = os.path.relpath(pt_file, "/kaggle/working")
        zf.write(pt_file, arcname)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Done — {zip_path} ({size_mb:.1f} MB)")

Zipping...
Done — /kaggle/working/RCTD_embeddings.zip (16.3 MB)
